In [ ]:
# Research Paper Answer Bot

## Project Overview

A Retrieval-Augmented Generation (RAG) system for answering questions from seminal Generative AI research papers.

### Technologies
- Python
- LangChain
- ChromaDB
- Ollama
- Nomic Embeddings
- BM25
- Cosine Similarity
- Gemma 3 1B
- Streamlit

### Dataset

4 GenAI Research Papers:
- LoRA: Low-Rank Adaptation of Large Language Models
- QLoRA: Efficient Finetuning of Quantized LLMs
- Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
- Attention Is All You Need

### Dataset Statistics
- Research Papers: 4
- Pages: 86
- Chunks: 649

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader

PDF_FOLDER = "data/papers"

TITLE_MAP = {
    "lora.pdf": "LoRA: Low-Rank Adaptation of Large Language Models",
    "qlora.pdf": "QLoRA: Efficient Finetuning of Quantized LLMs",
    "rag.pdf": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
    "researchpaper 1.pdf": "Attention Is All You Need"
}

documents = []

for file in os.listdir(PDF_FOLDER):
    if file.endswith(".pdf"):
        path = os.path.join(PDF_FOLDER, file)
        pages = PyPDFLoader(path).load()

        for page in pages:
            page.metadata["paper_title"] = TITLE_MAP.get(
                file,
                os.path.splitext(file)[0]
            )
            page.metadata["page_number"] = page.metadata.get("page", 0) + 1

        documents.extend(pages)

print("Total pages loaded:", len(documents))

C:\Users\kathi\AppData\Local\Temp\ipykernel_29692\1738200750.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total pages loaded: 86


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total pages:", len(documents))
print("Total chunks:", len(chunks))

Total pages: 86
Total chunks: 649


In [3]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vector = embeddings.embed_query(
    "What is LoRA in large language models?"
)

print("EMBEDDING CREATED SUCCESSFULLY!")
print("Embedding dimensions:", len(vector))

EMBEDDING CREATED SUCCESSFULLY!
Embedding dimensions: 768


In [4]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="research_papers",
    embedding_function=embeddings,
    persist_directory="vectorstore",
    collection_metadata={"hnsw:space": "cosine"}
)

batch_size = 25

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    vectorstore.add_documents(batch)
    print(f"Embedded {min(i + batch_size, len(chunks))}/{len(chunks)} chunks")

print("\nVECTOR DATABASE CREATED SUCCESSFULLY!")
print("Total chunks stored:", len(chunks))

Embedded 25/649 chunks
Embedded 50/649 chunks
Embedded 75/649 chunks
Embedded 100/649 chunks
Embedded 125/649 chunks
Embedded 150/649 chunks
Embedded 175/649 chunks
Embedded 200/649 chunks
Embedded 225/649 chunks
Embedded 250/649 chunks
Embedded 275/649 chunks
Embedded 300/649 chunks
Embedded 325/649 chunks
Embedded 350/649 chunks
Embedded 375/649 chunks
Embedded 400/649 chunks
Embedded 425/649 chunks
Embedded 450/649 chunks
Embedded 475/649 chunks
Embedded 500/649 chunks
Embedded 525/649 chunks
Embedded 550/649 chunks
Embedded 575/649 chunks
Embedded 600/649 chunks
Embedded 625/649 chunks
Embedded 649/649 chunks

VECTOR DATABASE CREATED SUCCESSFULLY!
Total chunks stored: 649


In [5]:
query = "What is LoRA and how does it reduce trainable parameters?"

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10
    }
)

results = retriever.invoke(query)

print("Query:", query)
print("\nTop Retrieved Sources:\n")

for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.metadata['paper_title']}")
    print(f"   Page: {doc.metadata['page_number']}")
    print(f"   Text: {doc.page_content[:200]}")
    print()

Query: What is LoRA and how does it reduce trainable parameters?

Top Retrieved Sources:

1. LoRA: Low-Rank Adaptation of Large Language Models
   Page: 1
   Text: rameters for downstream tasks. Compared to GPT-3 175B ﬁne-tuned with Adam,
LoRA can reduce the number of trainable parameters by 10,000 times and the
GPU memory requirement by 3 times. LoRA performs o

2. LoRA: Low-Rank Adaptation of Large Language Models
   Page: 1
   Text: rank-deﬁciency in language model adaptation, which sheds light on the efﬁcacy of
LoRA. We release a package that facilitates the integration of LoRA with PyTorch
models and provide our implementations

3. LoRA: Low-Rank Adaptation of Large Language Models
   Page: 21
   Text: MultiNLI, the combination of LoRA+PE doesn’t perform better than LoRA, possibly because LoRA
on its own already achieves performance comparable to the human baseline. Secondly, we notice
that LoRA+PL 



In [6]:
from rank_bm25 import BM25Okapi

texts = [doc.page_content for doc in chunks]

tokenized_texts = [
    text.lower().split()
    for text in texts
]

bm25 = BM25Okapi(tokenized_texts)

query = "What is LoRA and how does it reduce trainable parameters?"

query_tokens = query.lower().split()

scores = bm25.get_scores(query_tokens)

top_indices = scores.argsort()[-3:][::-1]

print("BM25 Top 3 Sources:\n")

for i, index in enumerate(top_indices, 1):
    doc = chunks[index]

    print(f"{i}. {doc.metadata['paper_title']}")
    print(f"   Page: {doc.metadata['page_number']}")
    print(f"   BM25 Score: {scores[index]:.4f}")
    print()

BM25 Top 3 Sources:

1. LoRA: Low-Rank Adaptation of Large Language Models
   Page: 10
   BM25 Score: 13.1966

2. LoRA: Low-Rank Adaptation of Large Language Models
   Page: 1
   BM25 Score: 12.5087

3. QLoRA: Efficient Finetuning of Quantized LLMs
   Page: 5
   BM25 Score: 11.9723



In [7]:
query = "What is LoRA and how does it reduce trainable parameters?"

dense_results = retriever.invoke(query)

bm25_scores = bm25.get_scores(query.lower().split())
top_indices = bm25_scores.argsort()[-3:][::-1]

bm25_results = [chunks[i] for i in top_indices]

hybrid_results = dense_results + bm25_results

unique_results = []
seen = set()

for doc in hybrid_results:
    key = (
        doc.metadata["paper_title"],
        doc.metadata["page_number"],
        doc.page_content[:100]
    )

    if key not in seen:
        seen.add(key)
        unique_results.append(doc)

print("Hybrid Retrieval Results:\n")

for i, doc in enumerate(unique_results[:5], 1):
    print(f"{i}. {doc.metadata['paper_title']}")
    print(f"   Page: {doc.metadata['page_number']}")
    print()

Hybrid Retrieval Results:

1. LoRA: Low-Rank Adaptation of Large Language Models
   Page: 1

2. LoRA: Low-Rank Adaptation of Large Language Models
   Page: 1

3. LoRA: Low-Rank Adaptation of Large Language Models
   Page: 21

4. LoRA: Low-Rank Adaptation of Large Language Models
   Page: 10

5. QLoRA: Efficient Finetuning of Quantized LLMs
   Page: 5



In [8]:
import numpy as np

query_vector = np.array(
    embeddings.embed_query(query)
)

reranked = []

for doc in unique_results:
    doc_vector = np.array(
        embeddings.embed_query(doc.page_content)
    )

    similarity = np.dot(query_vector, doc_vector) / (
        np.linalg.norm(query_vector) *
        np.linalg.norm(doc_vector)
    )

    reranked.append((doc, similarity))

reranked.sort(
    key=lambda x: x[1],
    reverse=True
)

print("Reranked Results:\n")

for i, (doc, score) in enumerate(reranked[:3], 1):
    print(f"{i}. {doc.metadata['paper_title']}")
    print(f"   Page: {doc.metadata['page_number']}")
    print(f"   Cosine Similarity: {score:.4f}")
    print()

Reranked Results:

1. LoRA: Low-Rank Adaptation of Large Language Models
   Page: 1
   Cosine Similarity: 0.7655

2. LoRA: Low-Rank Adaptation of Large Language Models
   Page: 1
   Cosine Similarity: 0.7638

3. LoRA: Low-Rank Adaptation of Large Language Models
   Page: 21
   Cosine Similarity: 0.7498



In [9]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma3:1b",
    temperature=0,
    num_predict=150
)

print("LLM LOADED SUCCESSFULLY!")

LLM LOADED SUCCESSFULLY!


In [10]:
context = "\n\n".join(
    [
        f"Paper: {doc.metadata['paper_title']}\n"
        f"Page: {doc.metadata['page_number']}\n"
        f"Content: {doc.page_content}"
        for doc, score in reranked[:3]
    ]
)

prompt = f"""
You are a research paper assistant.

Answer the question using ONLY the provided context.
If the answer is not available in the context, say:
"I could not find the answer in the provided papers."

Question:
{query}

Context:
{context}

Give a clear and concise answer.
"""

response = llm.invoke(prompt)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(response.content)

print("\nTOP 3 SOURCES:")
for i, (doc, score) in enumerate(reranked[:3], 1):
    print(
        f"{i}. {doc.metadata['paper_title']} "
        f"- Page {doc.metadata['page_number']}"
    )

QUESTION:
What is LoRA and how does it reduce trainable parameters?

ANSWER:
LoRA (Low-Rank Adaptation) is a technique that reduces trainable parameters by training only a small number of parameters. It achieves this by learning low-rank matrices instead of the full model parameters. This significantly reduces the computational cost and memory requirements compared to full fine-tuning.

TOP 3 SOURCES:
1. LoRA: Low-Rank Adaptation of Large Language Models - Page 1
2. LoRA: Low-Rank Adaptation of Large Language Models - Page 1
3. LoRA: Low-Rank Adaptation of Large Language Models - Page 21


In [11]:
test_questions = [
    "What is LoRA and how does it reduce trainable parameters?",
    "What is the main idea behind the Transformer architecture?",
    "What is Retrieval-Augmented Generation?",
    "How does RAG improve knowledge-intensive NLP tasks?",
    "What is QLoRA?",
    "How does QLoRA reduce memory requirements?",
    "What is the purpose of attention in the Transformer?",
    "What are the advantages of low-rank adaptation?",
    "How does LoRA differ from full fine-tuning?",
    "What problem does Retrieval-Augmented Generation solve?"
]

print("Total Test Questions:", len(test_questions))

for i, question in enumerate(test_questions, 1):
    print(f"{i}. {question}")

Total Test Questions: 10
1. What is LoRA and how does it reduce trainable parameters?
2. What is the main idea behind the Transformer architecture?
3. What is Retrieval-Augmented Generation?
4. How does RAG improve knowledge-intensive NLP tasks?
5. What is QLoRA?
6. How does QLoRA reduce memory requirements?
7. What is the purpose of attention in the Transformer?
8. What are the advantages of low-rank adaptation?
9. How does LoRA differ from full fine-tuning?
10. What problem does Retrieval-Augmented Generation solve?


In [12]:
evaluation_results = []

for question in test_questions:

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        [
            f"Paper: {doc.metadata['paper_title']}\n"
            f"Page: {doc.metadata['page_number']}\n"
            f"Content: {doc.page_content}"
            for doc in retrieved_docs[:3]
        ]
    )

    prompt = f"""
You are a research paper assistant.

Answer using ONLY the provided context.
If the answer is not available, say:
"I could not find the answer in the provided papers."

Question:
{question}

Context:
{context}
"""

    response = llm.invoke(prompt)

    sources = [
        f"{doc.metadata['paper_title']} - Page {doc.metadata['page_number']}"
        for doc in retrieved_docs[:3]
    ]

    evaluation_results.append({
        "question": question,
        "answer": response.content,
        "top_3_sources": sources
    })

print("Evaluation completed!")
print("Questions evaluated:", len(evaluation_results))

Evaluation completed!
Questions evaluated: 10


In [13]:
import pandas as pd

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,question,answer,top_3_sources
0,What is LoRA and how does it reduce trainable ...,LoRA (Low-Rank Adaptation) is a technique that...,[LoRA: Low-Rank Adaptation of Large Language M...
1,What is the main idea behind the Transformer a...,The main idea behind the Transformer architect...,"[Attention Is All You Need - Page 3, Attention..."
2,What is Retrieval-Augmented Generation?,Retrieval-Augmented Generation (RAG) is a gene...,[Retrieval-Augmented Generation for Knowledge-...
3,How does RAG improve knowledge-intensive NLP t...,RAG improves knowledge-intensive NLP tasks by ...,[Retrieval-Augmented Generation for Knowledge-...
4,What is QLoRA?,QLoRA is a technology that helps to close the ...,[QLoRA: Efficient Finetuning of Quantized LLMs...
5,How does QLoRA reduce memory requirements?,QLORA reduces memory requirements by automatic...,[QLoRA: Efficient Finetuning of Quantized LLMs...
6,What is the purpose of attention in the Transf...,The purpose of attention in the Transformer is...,"[Attention Is All You Need - Page 5, Attention..."
7,What are the advantages of low-rank adaptation?,LoRA (Low-Rank Adaptation) offers several adva...,[LoRA: Low-Rank Adaptation of Large Language M...
8,How does LoRA differ from full fine-tuning?,LoRA differs from full fine-tuning by using lo...,[QLoRA: Efficient Finetuning of Quantized LLMs...
9,What problem does Retrieval-Augmented Generati...,Retrieval-Augmented Generation solves the prob...,[Retrieval-Augmented Generation for Knowledge-...


In [14]:
evaluation_df.to_csv(
    "experiments/notebook_evaluation_results.csv",
    index=False
)

print("Evaluation results saved successfully!")

Evaluation results saved successfully!


In [15]:
from langchain_ollama import OllamaEmbeddings

query = "What is LoRA and how does it reduce trainable parameters?"

models = [
    "nomic-embed-text",
    "mxbai-embed-large"
]

for model in models:
    embedding_model = OllamaEmbeddings(model=model)
    vector = embedding_model.embed_query(query)

    print("Model:", model)
    print("Embedding Dimensions:", len(vector))
    print("First 5 Values:", vector[:5])
    print("-" * 50)

Model: nomic-embed-text
Embedding Dimensions: 768
First 5 Values: [0.015933197, 0.054841593, -0.16935667, -0.03559199, 0.068110056]
--------------------------------------------------
Model: mxbai-embed-large
Embedding Dimensions: 1024
First 5 Values: [0.013397918, 0.043037765, -0.010752155, -0.04282077, -0.025444118]
--------------------------------------------------


In [16]:
evaluation_criteria = {
    "Relevance": "Does the retrieved context directly relate to the question?",
    "Accuracy": "Does the generated answer correctly reflect the research paper content?",
    "Groundedness": "Is the answer supported by the retrieved context?"
}

for metric, definition in evaluation_criteria.items():
    print(f"{metric}: {definition}")

Relevance: Does the retrieved context directly relate to the question?
Accuracy: Does the generated answer correctly reflect the research paper content?
Groundedness: Is the answer supported by the retrieved context?


In [17]:
evaluation_scores = pd.DataFrame({
    "Question_ID": range(1, 11),
    "Relevance": [None] * 10,
    "Accuracy": [None] * 10,
    "Groundedness": [None] * 10
})

evaluation_scores

,Question_ID,Relevance,Accuracy,Groundedness
0,1,None,None,None
1,2,None,None,None
2,3,None,None,None
3,4,None,None,None
4,5,None,None,None
5,6,None,None,None
6,7,None,None,None
7,8,None,None,None
8,9,None,None,None
9,10,None,None,None


In [18]:
failure_question = "What is the architecture of a quantum computer?"

failure_results = retriever.invoke(failure_question)

context = "\n\n".join(
    [
        f"Paper: {doc.metadata['paper_title']}\n"
        f"Page: {doc.metadata['page_number']}\n"
        f"Content: {doc.page_content}"
        for doc in failure_results[:3]
    ]
)

failure_prompt = f"""
You are a research paper assistant.

Answer ONLY using the provided research paper context.
If the answer is not available in the context, say:
"I could not find the answer in the provided papers."

Question:
{failure_question}

Context:
{context}
"""

failure_response = llm.invoke(failure_prompt)

print("Failure Case Question:")
print(failure_question)

print("\nSystem Response:")
print(failure_response.content)

Failure Case Question:
What is the architecture of a quantum computer?

System Response:
The architecture of a quantum computer is a complex and evolving topic, but it generally involves multiple interconnected components. The provided context focuses on the key aspects of quantum computing, particularly in the context of quantum algorithms and hardware design. It doesn't provide a detailed architectural description.


In [19]:
strict_failure_prompt = f"""
You are a research paper question-answering assistant.

IMPORTANT RULES:
1. Use ONLY the provided context.
2. Do NOT use your own knowledge.
3. If the context does not contain enough information to answer the question,
   respond EXACTLY:
   "I could not find the answer in the provided papers."
4. Do not make assumptions or provide general explanations.

Question:
{failure_question}

Context:
{context}

Answer:
"""

strict_response = llm.invoke(strict_failure_prompt)

print("Failure Case Question:")
print(failure_question)

print("\nStrict RAG Response:")
print(strict_response.content)

Failure Case Question:
What is the architecture of a quantum computer?

Strict RAG Response:
I could not find the answer in the provided papers.
